# CYMEK FORMATION-MUX-001 — Kaggle T4 x2

## BEFORE RUNNING

Kaggle **Settings → Accelerator → GPU T4 x2** and **Internet → ON**.

This notebook runs the prospective pre-execution repair (Amendment 1). It does **not** use the failed S1 launcher. Official science requires exactly two visible T4 GPUs. The operator verifies frozen Science Commit S2 before training, runs qualification + measured calibration, automatically uses both T4s as independent matched-bundle workers, and partitions across Kaggle sessions only when the measured projection cannot safely fit one session.

If a partitioned run is required, save the Kaggle version/output, attach that output to the next run, and rerun this exact notebook. Completed arms are preserved and incomplete matched bundles remain pending.


In [ ]:
# CELL 1 — fetch and verify the immutable operator commit (do NOT checkout science directly)
import pathlib, subprocess, sys
REPO = pathlib.Path('/kaggle/working/An-Ra-the-new-AGI')
REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
OPERATOR_COMMIT = '878072f75821bd93d1533792d7e9b14c347306c5'
OPERATOR_PATH = 'tools/formation_mux_001_kaggle_operator_v3.py'
OPERATOR_BLOB = 'a7be737b1c85de5fa28a7e0c80c35fac86d1a55a'
SCIENCE_COMMIT_S2 = '534dcccfb8f96a30e80d71a30160e6a16eaa1ede'
if not REPO.exists():
    subprocess.run(['git', 'clone', REMOTE, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', OPERATOR_COMMIT], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert head == OPERATOR_COMMIT, (head, OPERATOR_COMMIT)
blob = subprocess.run(['git', '-C', str(REPO), 'hash-object', OPERATOR_PATH], capture_output=True, text=True, check=True).stdout.strip()
assert blob == OPERATOR_BLOB, (blob, OPERATOR_BLOB)
subprocess.run(['git', '-C', str(REPO), 'cat-file', '-e', SCIENCE_COMMIT_S2 + '^{commit}'], check=True)
try:
    import tokenizers  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers'], check=True)
print('OPERATOR VERIFIED:', OPERATOR_COMMIT)
print('SCIENCE S2 AVAILABLE:', SCIENCE_COMMIT_S2)


In [ ]:
# CELL 2 — one canonical run path
import subprocess, sys
code = subprocess.run([
    sys.executable, '-u', 'tools/formation_mux_001_kaggle_operator_v3.py',
    '--repo', '/kaggle/working/An-Ra-the-new-AGI',
    '--out', '/kaggle/working/FORMATION_MUX_001',
], cwd='/kaggle/working/An-Ra-the-new-AGI')
if code.returncode != 0:
    raise RuntimeError('FORMATION-MUX operator failed closed with exit ' + str(code.returncode) + '; inspect /kaggle/working/FORMATION_MUX_001/GLOBAL_FAILURE.json and the results ZIP')


In [ ]:
# CELL 3 — status / retrieval
import json, pathlib
root = pathlib.Path('/kaggle/working/FORMATION_MUX_001')
bundle = pathlib.Path('/kaggle/working/FORMATION_MUX_001_RESULTS.zip')
state = json.loads((root / 'CAMPAIGN_STATE.json').read_text()) if (root / 'CAMPAIGN_STATE.json').exists() else {}
print('STATUS:', state.get('status'))
print('ARMS:', state.get('complete_arms'), '/', state.get('required_arms'))
print('PENDING SEED BUNDLES:', state.get('pending_seed_bundles'))
for exp in ('CS-MECH-002', 'REP-FORM-003A'):
    p = root / exp / 'FINAL_RESULT.json'
    print(exp, '->', json.loads(p.read_text()).get('verdict') if p.exists() else 'not finalized')
print('RESULT ZIP:', bundle, 'exists=', bundle.exists())
if state.get('status') == 'PARTIAL_SESSION':
    print('NEXT: Save this Kaggle version/output, attach the output to the next run, and rerun this exact notebook.')
